# Search Agent

Notebook showing how to search and get item details

## Imports

In [1]:
import sys
sys.path.insert(0, "../src/")

In [2]:
import dotenv
import ipywidgets 
import rich

In [3]:
dotenv.load_dotenv("../.env")

True

In [4]:
import os
# os.environ["LANGCHAIN_TRACING_V2"] = "true"

In [29]:
import yaml
from pathlib import Path
import datetime

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.globals import set_debug
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
import tools
import agents

In [6]:
from google import genai


In [7]:
import models
from models import WalmartItem
import factory
import prompts
import utils

In [8]:
walmart_client = factory.create_walmart_api_client()

## Google Grounding

LLMs can hallucinate.
One way to prevent it is through RAG.

With some LLM providers, you can use their web search tool. 
Or u can use external tools.

Google's gemini model allows u to use google search as `grounding`. 

Given a query, googl's llm will make some web searches, and then use the search results as context.

### Demo of google grounding

In [9]:

model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")

model_with_search = model.bind_tools([{"google_search": {}}])
response = model_with_search.invoke("When is the next total solar eclipse in US?")



In [10]:
rich.print(response)

AIMessage(
    content=[
        {
            'type': 'text',
            'text': 'The next total solar eclipse visible in the United States depends on whether you are looking 
for any U.S. territory or specifically the "lower 48" states.\n\n### 1. The Next Total Solar Eclipse in the U.S. 
(Alaska Only)\n*   **Date:** March 30, 2033\n*   **Visibility:** Only visible in **Alaska**. The path of totality 
will cross over western and northern Alaska, including towns like Nome, Kotzebue, and Utqiaġvik (Barrow).\n\n### 2.
The Next Total Solar Eclipse in the Contiguous U.S. (Lower 48)\n*   **Date:** August 22, 2044\n*   **Visibility:** 
This eclipse will be visible in only three states: **Montana**, **North Dakota**, and **South Dakota**. It will 
occur in the late evening, near sunset. \n    *   *Note: You may see this referred to as the "August 23 eclipse" in
some scientific databases because that is the date in Universal Time (UTC), but it falls on August 22 in U.S. time 
zones.*\n\n### 3. The Next "Coast-to-Coast" Total Solar Eclipse\n*   **Date:** August 12, 2045\n*   **Visibility:**
This will be a massive event similar to the 2017 and 2024 eclipses. The path of totality will span from 
**California to Florida**, passing through states including Nevada, Utah, Colorado, Kansas, Oklahoma, Arkansas, 
Mississippi, Alabama, and Georgia.\n\n**Summary of upcoming solar events in the U.S.:**\n| Date | Type | Location 
|\n| :--- | :--- | :--- |\n| **August 12, 2026** | Partial Solar Eclipse | Most of the U.S. (Total in 
Greenland/Spain) |\n| **January 26, 2028** | Partial Solar Eclipse | Parts of the U.S. |\n| **March 30, 2033** | 
**Total Solar Eclipse** | **Alaska** |\n| **August 22, 2044** | **Total Solar Eclipse** | **MT, ND, SD** |\n| 
**August 12, 2045** | **Total Solar Eclipse** | **CA to FL (Coast-to-Coast)** |',
            'extras': {
                'signature': 
'EugFCuUFAXLI2nzkZ54iSbjUQ+d+npntLZdxaoEi9OTyGNeH1YOZQhxclD4Rbq/f6BOWS1LNwISZ4pyfoqgPPDAAn/BSpue9C5KUGvb4sjPLWuO49u
hAegXCsu4g2X97bav7FkX0h6Ayrb3sbXuHLiQSitywrs4H3CmveNDa3oEvauIXgKnFtnfNAEoscy1RIaTmSOIuQmKjeimcXL3PeBGlXaxg7Wi1M8ITX
w4jNs0rzUA9sJ1ZBVlnr3c7lZi9SESIxM4pa9BxAER2VfdT9g8gXS0JRiHY8sBIqZtgoMcnOrHRLio5k1+R6Ev3GVm5Lr5Wh3OdGR9Tw1Oxz/qpvcMO
Q+YTfaZ6MpmIh3FzjDuKOUs+WNAfMIyyG/B/SCGyUyZ8zsiG9LIwMshYq12ZYQ7CgS2yyvSSgDeQL38zglf/o9EwF2iUSXEStr/0icqFQJQPWezE6qy
TK0Uoj1fH+mnnn+agZifkPCt200LJpIG/ZcbD1F5UY3XY61gKmlLvyrxMO9q7qSnxus9AQakGb8zJAETTp+i/SWO25/OySdUmq6g4xuSZVBbyR+B2SW
rMDlx9mefzQG8sLgtB0m05BwS+D7Lf1MJyoifE30XG7H/c5kf+rPtoU6ZePLWnj0VcSJ72EhAHHSlTm1Tt8Xu3sqqYYAfI9njAeYH+/PYsp58iIa/qi
hndpnUIN5sPrbHXMqQdRSpMabgtMe5oSFqWA/SI0setniniMcHTQD/CmNpRmuEiTySX8bmJo8DTQarfWi6DGq3XskOGNieP4f6JtoyhCOxrpvsNJYFz
FDwaR7QtJuqGlvYYlq9bOUJhQlMF3424zW529zAZqRMAzKIQsg8KdpSvSSBfb3gNFM428rg74JmpN7ePJ3DQlrK+RO8YGPs/w/YiSm4/A6tr3SspT2Q
vthPrmu35JdYKSJYukcE6y3wJ7cRDhDDNeiGVIIQXCP0ulx/UkRW+OcI8DmAEZGNv2s7J/BVcwb3i'
            }
        }
    ],
    additional_kwargs={},
    response_metadata={
        'finish_reason': 'STOP',
        'model_name': 'gemini-3-flash-preview',
        'safety_ratings': [],
        'grounding_metadata': {
            'google_maps_widget_context_token': None,
            'grounding_chunks': [
                {
                    'maps': None,
                    'retrieved_context': None,
                    'web': {
                        'domain': None,
                        'title': 'forbes.com',
                        'uri': 
'https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEIQDiGYQBgYWUC2Ew2WVOuBU04_pqng3-NWxboje6W12M
wNRrysM6PuSOWjphGCXln-9U-AXSUCDoD5qbwyDhUZTk1M4Sky3n4KLlES252YlzjmZl6VhCMG8I4h08ucJoBYe7ti_F57DXCxUcbW7oy3Di_L5N-4f
hQjGzhxpwVsVs3raME9I4yJ2YAFcnUz6I01-l2XjjGttg8AXj20EGCqZhKjtKjG-iVZVxjgxVMocTgPvE5'
                    }
                },
                {
                    'maps': None,
                    'retrieved_context': None,
                    'web': {
                        'domain': None,
            

Some key fields in reponse. 

`web_search_queries` : list of queries that google used

`grounding_chunks`: web sites that are used as references

For part of the output, it indicates support for that statement using `grounding_chunk_indices`

## Ecommerce Product Research

In [11]:
query_mothers_day = "What are popular gifts for Mother's Day?"

In [12]:
utils.display_code(models.ProductResearchResponse,"ProductResearchResponse")

╭────────────────────────────────────── ProductResearchResponse ──────────────────────────────────────╮
│   1 class ProductResearchResponse(BaseModel):                                                       │
│   2     normalized_intent: str = Field(description="One-line interpretation of the user's request") │
│   3     clarifying_questions: list[str] = Field(description="2–4 questions that reduce ambiguity")  │
│   4     query_candidates: list[SearchQuery] = Field(description="Exactly 5 query candidates")       │
│   5                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [13]:
utils.display_code(models.SearchQuery,"SearchQuery")

╭────────────────────────────────────────────────── SearchQuery ──────────────────────────────────────────────────╮
│   1 class SearchQuery(BaseModel):                                                                               │
│   2     query: str = Field(description="E-commerce search-bar query. Prefer <= 6 words, no sentences.")         │
│   3     query_type: Literal["broad", "specific"] = Field(description="3 broad + 2 specific")                    │
│   4     category: Optional[str] = Field(default=None, description="Approx catalog category/vertical")           │
│   5     rationale: str = Field(description="Short justification grounded in search results (no year/trend claim │
│   6                                                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

#### Using Lancchain's built-in Google Search tool

there is a bug with langchain binding to google search . 

In [14]:






# parser_product_research_response = PydanticOutputParser(pydantic_object=models.ProductResearchResponse)

# # doesn't seem to work. with tool calls and structured output not captured properly
# def product_research_langchain(query: str):
#     """Map a user query to ecommerce search queries using Google Search tool."""

    
#     instruction = """
#     {system_prompt}

#     format_instructions: {format_instructions}
#     """.format(system_prompt=prompts.PROMPT_PRODUCT_RESEARCH, format_instructions=parser_product_research_response.get_format_instructions())

#     model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview",include_thoughts=False,  ).bind_tools([
#         {"google_search": {}}
#         ])
    
#     # NOTE: this doesn't work. it doesn't seem to use google langchain tool call
#     #model = model.with_structured_output(ProductSearchResponse)

#     messages = [{"role": "system", "content": instruction}, {"role": "user", "content": query}]

#     response_structured = model.invoke(messages)
#     return response_structured

In [15]:
# response = product_research_langchain(query_mothers_day)
# rich.print(response)

In [16]:
# product_researchresponse:models.ProductResearchResponse = parser_product_research_response.parse(response.text)

#### Using Google LLM Library 

In [20]:
utils.display_code(tools.product_research_llm_call,"Product Research LLM Call")

╭───────────────────────────── Product Research LLM Call ──────────────────────────────╮
│    1 def product_research_llm_call(query: str) -> models.ProductResearchResponse:    │
│    2     client = genai.Client()                                                     │
│    3                                                                                 │
│    4     response = client.models.generate_content(                                  │
│    5         model="gemini-3-flash-preview",                                         │
│    6         contents=query,                                                         │
│    7         config=genai.types.GenerateContentConfig(                               │
│    8             tools=[                                                             │
│    9                 {"google_search": {}}                                           │
│   10             ],                                                                  │
│   11             system_instruction=prompts.PROMPT_PRODUCT_RESEARCH,                 │
│   12             response_mime_type="application/json",                              │
│   13             response_schema=models.ProductResearchResponse.model_json_schema(), │
│   14             )                                                                   │
│   15     )                                                                           │
│   16                                                                                 │
│   17     return response                                                             │
│   18                                                                                 │
╰──────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
response = tools.product_research_tool.func(query_mothers_day)
rich.print(response)

product_research_response = models.ProductResearchResponse.model_validate_json(response.text)
rich.print(product_research_response)

lets wrap the llm call and parsing into a single function

In [21]:
utils.display_code(tools.product_research_tool.func,"Product Research Tool")

╭──────────────────────────────────────── Product Research Tool ────────────────────────────────────────╮
│   1 @tool                                                                                             │
│   2 def product_research_tool(query:str) -> models.ProductResearchResponse:                           │
│   3     """Perform research to map a user question into ecommerce queries"""                          │
│   4                                                                                                   │
│   5     response = product_research_llm_call(query)                                                   │
│   6     product_research_response = models.ProductResearchResponse.model_validate_json(response.text) │
│   7                                                                                                   │
│   8     return product_research_response                                                              │
│   9                                                                                                   │
╰───────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Agent

In [22]:
from langchain.agents import create_agent
from langchain.tools import tool

In [23]:
from models import WalmartItem
import tools

Search Walmart Tool just searches walmart for items based on a query and returns top n items 

In [25]:
utils.display_code(tools.search_walmart_tool.func,"Search Walmart Tool")

╭─────────────────────────────────────────── Search Walmart Tool ───────────────────────────────────────────╮
│    1 @tool                                                                                                │
│    2 def search_walmart_tool(queries:list[str], top_n_items_per_query:int=3) -> list[models.WalmartItem]: │
│    3     """Search Walmart for the given customer queries ."""                                            │
│    4                                                                                                      │
│    5     all_items = []                                                                                   │
│    6     all_facets = []                                                                                  │
│    7     seen_item_ids = set()                                                                            │
│    8     seen_facet_names = set()                                                                         │
│    9                                                                                                      │
│   10     for query in queries:                                                                            │
│   11                                                                                                      │
│   12         items, facets = walmart_client.search(query=query)                                           │
│   13                                                                                                      │
│   14         for idx, item in enumerate(items):                                                           │
│   15             if idx < top_n_items_per_query:                                                          │
│   16                 if item.item_id not in seen_item_ids:                                                │
│   17                     all_items.append(item)                                                           │
│   18                     seen_item_ids.add(item.item_id)                                                  │
│   19         for idx, facet in enumerate(facets):                                                         │
│   20             if idx < top_n_items_per_query:                                                          │
│   21                 if facet.name not in seen_facet_names:                                               │
│   22                     all_facets.append(facet)                                                         │
│   23                     seen_facet_names.add(facet.name)                                                 │
│   24                                                                                                      │
│   25                                                                                                      │
│   26                                                                                                      │
│   27     return all_items                                                                                 │
│   28                                                                                                      │
╰───────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [27]:
utils.display_code(tools.product_details_tool.func,"Product Details Tool")

╭─────────────────────────── Product Details Tool ───────────────────────────╮
│   1 @tool                                                                  │
│   2 def product_details_tool(item_id:str) -> models.WalmartProductDetails: │
│   3     """Get Details for a Walmart item by item id"""                    │
│   4                                                                        │
│   5     response = walmart_client.product_details(item_id=item_id)         │
│   6                                                                        │
│   7     return response                                                    │
│   8                                                                        │
╰────────────────────────────────────────────────────────────────────────────╯

In [33]:
query = "What are popular gifts for Mother's Day?"


In [34]:
items, facets = walmart_client.search(query=query_mothers_day)

In [35]:
tools.search_walmart_tool.invoke( {'queries':[query] })

[WalmartItem(item_id='2009728644', title='Preserved Rose Flower with Heart Necklace, Eternal Purple Real Rose with Music LED Lights,Gifts for Her Women Mom Wife Grandma on Anniversary Birthday Valentines, Purple', image='https://i5.walmartimages.com/seo/Emibele-Preserved-Rose-Flower-Heart-Necklace-Eternal-Purple-Real-Rose-Music-LED-Lights-Women-Wife-Grandma-Anniversary-Birthday-Romantic-Gifts-Purple_e557734a-7201-4f05-9b6b-86b2a7cbb446.4f98568bb6a8b2b3c0f79c952cb3599f.jpeg?odnHeight=180&odnWidth=180&odnBg=FFFFFF', url='/ip/Emibele-Preserved-Rose-Flower-Heart-Necklace-Eternal-Purple-Real-Rose-Music-LED-Lights-Women-Wife-Grandma-Anniversary-Birthday-Romantic-Gifts-Purple/2009728644?classType=VARIANT&athbdg=L1600', link='https://www.walmart.com/ip/Emibele-Preserved-Rose-Flower-Heart-Necklace-Eternal-Purple-Real-Rose-Music-LED-Lights-Women-Wife-Grandma-Anniversary-Birthday-Romantic-Gifts-Purple/2009728644?classType=VARIANT&athbdg=L1600', price=28.99, rating_score=4.7, rating_count=174),
 W

In [36]:
rich.print(items[0])

WalmartItem(
    item_id='2009728644',
    title='Preserved Rose Flower with Heart Necklace, Eternal Purple Real Rose with Music LED Lights,Gifts for Her 
Women Mom Wife Grandma on Anniversary Birthday Valentines, Purple',
    image='https://i5.walmartimages.com/seo/Emibele-Preserved-Rose-Flower-Heart-Necklace-Eternal-Purple-Real-Rose-M
usic-LED-Lights-Women-Wife-Grandma-Anniversary-Birthday-Romantic-Gifts-Purple_e557734a-7201-4f05-9b6b-86b2a7cbb446.
4f98568bb6a8b2b3c0f79c952cb3599f.jpeg?odnHeight=180&odnWidth=180&odnBg=FFFFFF',
    url='/ip/Emibele-Preserved-Rose-Flower-Heart-Necklace-Eternal-Purple-Real-Rose-Music-LED-Lights-Women-Wife-Gran
dma-Anniversary-Birthday-Romantic-Gifts-Purple/2009728644?classType=VARIANT&athbdg=L1600',
    link='https://www.walmart.com/ip/Emibele-Preserved-Rose-Flower-Heart-Necklace-Eternal-Purple-Real-Rose-Music-LE
D-Lights-Women-Wife-Grandma-Anniversary-Birthday-Romantic-Gifts-Purple/2009728644?classType=VARIANT&athbdg=L1600',
    price=28.99,
    rating_score=4.7,
    rating_count=174
)

In [37]:
item_id = items[0].item_id
item_id='2009728644'
res = tools.product_details_tool.invoke({'item_id': item_id})

rich.print(res)

WalmartProductDetails(
    item_id='2009728644',
    title='Emibele Preserved Rose Flower with Heart Necklace, Eternal Purple Real Rose with Music LED Lights,Gifts 
for Her Women Mom Wife Grandma on Anniversary Birthday Valentines, Purple',
    url='https://www.walmart.com/ip/Emibele-Preserved-Rose-Flower-Heart-Necklace-Eternal-Purple-Real-Rose-Music-LED
-Lights-Women-Wife-Grandma-Anniversary-Birthday-Romantic-Gifts-Purple/2009728644',
    brand='Emibele',
    description='About this item Product details Preserved Real Rose with "I Love You" Necklace There isn\'t much 
more of a romantic gesture than sending an eternal rose to someone. With a fancy necklace saying "I love you" in 
100 different languages, beautiful melodies and warm lights, these stunning beauties would always help to touch the
heart of your recipients. Unisex Gift 🌹Valentine\'s Day Gifts? Need a special gift for this Valentine\'s Day? 
Instead of those easily wilted flowers, try this preserved rose box with a gold plated necklace as a perfect gift 
for your girlfriend, expressing your forever love for her. 🌹Mother\'s Day Gifts? Look for an affectionate gift for
Mother\'s Day? Rather than flashy bouquets, choose this eternal rose and "I love you" necklace as a wonderful gift 
to deliver your endless love and gratitude for your mother, grandmother, or aunt... 🌹Birthday Gifts? Want to give 
someone an unforgettable birthday gift? Different from those normal surprises, this amazing forever rose and 
necklace gift set will send your true blessing and make a "wow" to your friend, girlfriend, or sister... 
🌹Anniversary Gifts? Search for the most incredible anniversary gift ever in life? Select this "I love you forever"
eternal rose and necklace set as a wedding anniversary gift for your wife, telling her that you will always love 
her and be her side. 🎁 Care Tips： Do not water the rose. Avoid direct sunlight and humidity Do not remove the 
rose from its base, it\'s secured and happy there. Avoid exposing the necklace to strong heat and all types of 
liquids. Clean the necklace regularly with a soft cloth. Store the product in a cool and dry spot. Key Features： 
🌹Preserved Rose: Preserved purple rose made of 100% natural fresh rose through a series of preservation method. It
can be stored for about 3-5 years without fading or withering. Eternal flower, forever love, stocking stuffers for 
women, christmas gifts for mom wife girlfriend, i love you gifts for her. 🌹LED Music Rose Gift Box: The 
double-door gift box hides the romance. When she open the eternal flower gift box, she will see beautiful rose with
led lighting, hear the light music "always with me". What a great surprise! Your loved ones will have a good mood 
whenever she looks at it! 🌹100 Languages I Love You Necklace: The elegant heart shape necklace is made of 
gold-plated copper with fancy crystal pendant. This love necklace used special engraving technology to engrave "I 
Love You" in 100 different languages into the love crystal stone. Bring the stone in the middle of the necklace 
close to your eyes or to your phone\'s rear camera and then you can catch words. 🎁Romantic Gift for Her: It\'s a 
perfect gift for her like girlfriend, wife, mom, grandma or your beloved ones as Valentines gifts, Mother\'s day 
gifts, Christmas gifts, Thanksgiving gifts, Birthday gifts, Wedding, Anniversary gifts. 🎁Package and Service: 
Luxurious & Exquisite suit eternal rose gift set includes: 1 Preserved Real Rose + 1 Rose Gift Box + 1 Heart 
Necklace + 1 Gift Bag + 1 Greeting Card. If you have any question about our product, please be free to contact us, 
our service team will answer in 24 hours and offer satisfactory solution. info: We aim to show you accurate product
information. Manufacturers, suppliers and others provide what you see here, and we have not verified it. See our 
disclaimer',
    specifications={
        'Material': 'Gold-Plated Copper',
        'Inscription': 'I Love You',
        'Retail packaging': 'G

## Shopping Agent with Web Search

In [38]:
product_agent  = agents.product_agent_setup()

In [39]:
utils.display_code(agents.product_agent_setup,"Product Agent")

╭─────────────────────────────────────────────── Product Agent ────────────────────────────────────────────────╮
│    1 def product_agent_setup():                                                                              │
│    2     """Sets up the shopping product agent with tools and memory."""                                     │
│    3                                                                                                         │
│    4     instruction = f'''                                                                                  │
│    5     You are a Walmart Ecommerce Sub Agent.                                                              │
│    6     You have two functionalities:                                                                       │
│    7     1. Help users find products on Walmart given their queries.                                         │
│    8     2. Helps answer detailed questions about a specific product given its Walmart item id.              │
│    9                                                                                                         │
│   10     When users asks you an ambigious query, you will follow the steps:                                  │
│   11     1. Call product_research_tool to map question to ecommerce quereis                                  │
│   12     2. Call search_walmart_tool with the generated queries                                              │
│   13                                                                                                         │
│   14     When users asks you a specific question about a product, you will:                                  │
│   15     1. Call product_details_tool with the Walmart item id to get product details if needed              │
│   16     2. Provide succint answer based on the product details and the users question.                      │
│   17                                                                                                         │
│   18     '''                                                                                                 │
│   19                                                                                                         │
│   20                                                                                                         │
│   21     shopping_tools=[tools.search_walmart_tool, tools.product_research_tool, tools.product_details_tool] │
│   22                                                                                                         │
│   23                                                                                                         │
│   24     shopping_agent = create_agent(                                                                      │
│   25         ChatOpenAI(model="gpt-4o-mini"),                                                                │
│   26         tools=shopping_tools,                                                                           │
│   27         checkpointer=memory,                                                                            │
│   28         debug=True,                                                                                     │
│   29         system_prompt=instruction                                                                       │
│   30     )                                                                                                   │
│   31                                                                                                         │
│   32     return shopping_agent                                                                               │
│   33                                                                                                         │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [40]:
config = {"configurable": {"thread_id": "product_agent_test"}}


resp = product_agent.invoke({"messages": [{"role": "user", "content": query }]}, config=config)

[values] {'messages': [HumanMessage(content="What are popular gifts for Mother's Day?", additional_kwargs={}, response_metadata={}, id='4229bb7c-6833-47ad-9f24-151d67a8ac97')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 275, 'total_tokens': 295, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-CvYfYWyq4iHGycLVYRPaTigGNhT0q', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b9b1e-194c-7023-8e89-f7768be26ff1-0', tool_calls=[{'name': 'product_research_tool', 'args': {'query': "popular gifts for Mother's Day"}, 'id': 'call_WsAHuyklC8dAse3kGqiQmDKj'

In [42]:
rich.print(resp['messages'] )

[
    HumanMessage(
        content="What are popular gifts for Mother's Day?",
        additional_kwargs={},
        response_metadata={},
        id='4229bb7c-6833-47ad-9f24-151d67a8ac97'
    ),
    AIMessage(
        content='',
        additional_kwargs={'refusal': None},
        response_metadata={
            'token_usage': {
                'completion_tokens': 20,
                'prompt_tokens': 275,
                'total_tokens': 295,
                'completion_tokens_details': {
                    'accepted_prediction_tokens': 0,
                    'audio_tokens': 0,
                    'reasoning_tokens': 0,
                    'rejected_prediction_tokens': 0
                },
                'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
            },
            'model_provider': 'openai',
            'model_name': 'gpt-4o-mini-2024-07-18',
            'system_fingerprint': 'fp_c4585b5b9c',
            'id': 'chatcmpl-CvYfYWyq4iHGycLVYRPaTigGNhT0q',
            'service_tier': 'default',
            'finish_reason': 'tool_calls',
            'logprobs': None
        },
        id='lc_run--019b9b1e-194c-7023-8e89-f7768be26ff1-0',
        tool_calls=[
            {
                'name': 'product_research_tool',
                'args': {'query': "popular gifts for Mother's Day"},
                'id': 'call_WsAHuyklC8dAse3kGqiQmDKj',
                'type': 'tool_call'
            }
        ],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 275,
            'output_tokens': 20,
            'total_tokens': 295,
            'input_token_details': {'audio': 0, 'cache_read': 0},
            'output_token_details': {'audio': 0, 'reasoning': 0}
        }
    ),
    ToolMessage(
        content='normalized_intent="Discover popular and trending Mother\'s Day gift ideas across wellness, 
jewelry, tech, and gourmet categories for 2026." clarifying_questions=[\'What is your approximate budget for the 
gift?\', \'Does the recipient have a specific interest, such as fitness, gardening, or home decor?\', \'Are you 
looking for a personalized gift that might require extra processing time?\', \'Do you prefer a physical product or 
a subscription-based service?\'] query_candidates=[SearchQuery(query="Mother\'s Day spa gift sets", 
query_type=\'broad\', category=\'Beauty & Personal Care\', rationale="Wellness and \'at-home spa\' kits are 
identified as a leading demand category for 2026 (source: paperart.com.cn)."), SearchQuery(query=\'personalized 
jewelry for mom\', query_type=\'broad\', category=\'Jewelry\', rationale=\'Sentimental and personalized items like 
birth-flower necklaces are trending as high-value emotional gifts (source: rushlet.blog).\'), 
SearchQuery(query=\'gourmet food gift baskets\', query_type=\'broad\', category=\'Grocery & Gourmet Food\', 
rationale="Gourmet hampers and luxury food treats are described as a \'raving\' trend for the 2026 season (source: 
giftstoindia24x7.com)."), SearchQuery(query=\'wifi digital picture frame\', query_type=\'specific\', 
category=\'Electronics\', rationale=\'Tech gifts that foster family connection, specifically digital frames like 
Skylight, are top recommendations for 2026 (source: rushlet.blog).\'), SearchQuery(query=\'smart ring health 
tracker\', query_type=\'specific\', category=\'Wearable Technology\', rationale=\'Smart rings are projected as the 
next evolution of wearable tech for women in 2026, blending fashion with wellness monitoring (source: 
truegiftworld.com).\')]',
        name='product_research_tool',
        id='bd63fa3f-6d1c-4ab8-8ec3-c2a053c4bd25',
        tool_call_id='call_WsAHuyklC8dAse3kGqiQmDKj'
    ),
    AIMessage(
        content='',
        additional_kwargs={'refusal': None},
        response_metadata={
            'token_usage': {
                'completion_tokens': 45,
                'prompt_tokens': 666,
                'total_tokens': 711,
                'completion_tokens_details

In [43]:
rich.print(resp['messages'][-1].content )

Here are some popular gift ideas for Mother's Day:

### Spa Gift Sets
1. **[Spa Bath Gift Sets for 
Women](https://www.walmart.com/ip/Spa-Bath-Gift-Sets-for-Women-8-Pcs-Rose-Scent-Gift-Baskets-for-Mom-Grandma-Sister
-Valentines-Day-Relaxing-Birthday-Gifts-for-Her/5438833867) - $17.99**
   ![Spa Bath Gift 
Sets](https://i5.walmartimages.com/seo/Spa-Bath-Gift-Sets-for-Women-8-Pcs-Rose-Scent-Gift-Baskets-for-Mom-Grandma-S
ister-Valentines-Day-Relaxing-Birthday-Gifts-for-Her_fb7d9f5e-4c6f-4bb5-8401-1642ecd78e09.db9344e04ffefdfbd7b6fd8de
e9682b1.jpeg)
   - Rating: 4.5 (90 reviews)

2. **[Gifts for Women Unique Spa Gifts Basket 
Set](https://www.walmart.com/ip/Gifts-Women-Unique-Spa-Gifts-Basket-Set-Birthday-Mothers-Day-Valentines-Day-Christm
as-Gifts-Mom-Sister-Her-Girlfriend-Wife-Bright-Orange-Gift-Box/17961254438) - $62.97**
   ![Unique Spa Gifts 
Basket](https://i5.walmartimages.com/seo/Gifts-Women-Unique-Spa-Gifts-Basket-Set-Birthday-Mothers-Day-Valentines-Da
y-Christmas-Gifts-Mom-Sister-Her-Girlfriend-Wife-Bright-Orange-Gift-Box_e9aae138-a230-4796-93c4-aac7e738cd48.95960c
9e1330efde5df04ef43762bc71.jpeg)

### Personalized Jewelry
1. **[18K White Gold Plated Love Heart Initial 
Necklace](https://www.walmart.com/ip/APSVO-18K-White-Gold-Plated-Love-Heart-Initial-Cross-Necklace-Women-Mom-Wife-C
Z-Hypoallergenic-S925-Sterling-Silver-Letter-Necklace-Birthday-Mother-s/16277451748) - $14.87**
   ![Heart Initial 
Necklace](https://i5.walmartimages.com/seo/APSVO-18K-White-Gold-Plated-Love-Heart-Initial-Cross-Necklace-Women-Mom-
Wife-CZ-Hypoallergenic-S925-Sterling-Silver-Letter-Necklace-Birthday-Mother-s_21ff1564-b502-4cf0-82e6-eaeb7bc13cf4.
876c72617d46b439145804c739b003e6.jpeg)
   - Rating: 4.4 (530 reviews)

### Gourmet Food Gift Baskets
1. **[GreatFoods Thank You Fresh Fruit & Sweets Gift 
Box](https://www.walmart.com/ip/GreatFoods-Thank-You-Fresh-Fruit-Sweets-Gift-Box-Gummie-Rings-Sea-Salt-Caramel-Trai
l-Mix-More-Fruit-Basket-Gift-Corporate-Gifting-Snack-Gift-Basket-G/10612765382) - $39.99**
   ![Fruit & Sweets Gift 
Box](https://i5.walmartimages.com/seo/GreatFoods-Thank-You-Fresh-Fruit-Sweets-Gift-Box-Gummie-Rings-Sea-Salt-Carame
l-Trail-Mix-More-Fruit-Basket-Gift-Corporate-Gifting-Snack-Gift-Basket-G_01559ced-ddcf-45f8-90ca-1755b3a12a7b.8e5a7
929511ddd7cf58193fef9e94862.jpeg)
   - Rating: 4.2 (16 reviews)

### Digital Picture Frames
1. **[10.1" HD Wifi Digital Photo Frame by 
Aura](https://www.walmart.com/ip/10-1-HD-Wifi-Carver-Mat-Black-Digital-Photo-Frame-by-Aura/17813968000) - $179.99**
   ![Wifi Digital Photo 
Frame](https://i5.walmartimages.com/seo/10-1-HD-Wifi-Carver-Mat-Black-Digital-Photo-Frame-by-Aura_986b594a-5afd-4b5
c-802f-5bf4ddcd06a2.2f75f834c5b9064e8a651c5c5a006ae8.jpeg)
   - Rating: 4.9 (2098 reviews)

### Smart Wearables
1. **[Smart Ring, Black, Size 9](https://www.walmart.com/ip/LivUp-Smart-Ring-Black-Size-9/17062503225) - $49.98**
   ![Smart 
Ring](https://i5.walmartimages.com/seo/LivUp-Smart-Ring-Black-Size-9_7305a80d-eddb-4d8d-85eb-e5d9353f832b.e3227f166
ff591f5c199a0d80e250965.jpeg)
   - Rating: 4.8 (482 reviews)

These gift ideas cover a range of interests and budgets, making it easier to find the perfect gift for Mother's 
Day!

In [44]:
resp = product_agent.invoke({"messages": [{"role": "user", "content": "tell me about the spa gift basket" }]}, config=config)


[values] {'messages': [HumanMessage(content="What are popular gifts for Mother's Day?", additional_kwargs={}, response_metadata={}, id='4229bb7c-6833-47ad-9f24-151d67a8ac97'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 275, 'total_tokens': 295, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-CvYfYWyq4iHGycLVYRPaTigGNhT0q', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b9b1e-194c-7023-8e89-f7768be26ff1-0', tool_calls=[{'name': 'product_research_tool', 'args': {'query': "popular gifts for Mother's Day"}, 'id': 'call_WsAHuyklC8dAse3kGqiQmDKj', 'type': 'tool_call'}], invalid_to

In [45]:
rich.print(resp['messages'][-1] )

AIMessage(
    content="It seems that I encountered an issue retrieving detailed information about the spa gift basket. 
However, I can provide you with some general information based on what I typically find:\n\n### Spa Bath Gift Sets 
for Women\n- **Price**: $17.99\n- **Description**: Generally, these spa gift sets include various items such as 
bath bombs, scented candles, body lotions, and other pampering essentials designed to create a relaxing spa-like 
experience at home. They are often beautifully packaged, making them excellent gifts for occasions like Mother's 
Day, birthdays, or holidays.\n- **Rating**: 4.5 stars based on 90 reviews, indicating a generally positive 
reception.\n\nYou can view the gift set directly 
[here](https://www.walmart.com/ip/Spa-Bath-Gift-Sets-for-Women-8-Pcs-Rose-Scent-Gift-Baskets-for-Mom-Grandma-Sister
-Valentines-Day-Relaxing-Birthday-Gifts-for-Her/5438833867).\n\nIf you have any specific questions or need more 
information, feel free to ask!",
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 220,
            'prompt_tokens': 6254,
            'total_tokens': 6474,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 0,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 6144}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-4o-mini-2024-07-18',
        'system_fingerprint': 'fp_c4585b5b9c',
        'id': 'chatcmpl-CvYpzl4ZnK5LXDwG2hHEZSmNeC0U1',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019b9b27-fa40-7cf2-bcc9-7c952acc2ee4-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 6254,
        'output_tokens': 220,
        'total_tokens': 6474,
        'input_token_details': {'audio': 0, 'cache_read': 6144},
        'output_token_details': {'audio': 0, 'reasoning': 0}
    }
)

In [46]:
rich.print(resp)

{
    'messages': [
        HumanMessage(
            content="What are popular gifts for Mother's Day?",
            additional_kwargs={},
            response_metadata={},
            id='4229bb7c-6833-47ad-9f24-151d67a8ac97'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 20,
                    'prompt_tokens': 275,
                    'total_tokens': 295,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-4o-mini-2024-07-18',
                'system_fingerprint': 'fp_c4585b5b9c',
                'id': 'chatcmpl-CvYfYWyq4iHGycLVYRPaTigGNhT0q',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019b9b1e-194c-7023-8e89-f7768be26ff1-0',
            tool_calls=[
                {
                    'name': 'product_research_tool',
                    'args': {'query': "popular gifts for Mother's Day"},
                    'id': 'call_WsAHuyklC8dAse3kGqiQmDKj',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 275,
                'output_tokens': 20,
                'total_tokens': 295,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='normalized_intent="Discover popular and trending Mother\'s Day gift ideas across wellness, 
jewelry, tech, and gourmet categories for 2026." clarifying_questions=[\'What is your approximate budget for the 
gift?\', \'Does the recipient have a specific interest, such as fitness, gardening, or home decor?\', \'Are you 
looking for a personalized gift that might require extra processing time?\', \'Do you prefer a physical product or 
a subscription-based service?\'] query_candidates=[SearchQuery(query="Mother\'s Day spa gift sets", 
query_type=\'broad\', category=\'Beauty & Personal Care\', rationale="Wellness and \'at-home spa\' kits are 
identified as a leading demand category for 2026 (source: paperart.com.cn)."), SearchQuery(query=\'personalized 
jewelry for mom\', query_type=\'broad\', category=\'Jewelry\', rationale=\'Sentimental and personalized items like 
birth-flower necklaces are trending as high-value emotional gifts (source: rushlet.blog).\'), 
SearchQuery(query=\'gourmet food gift baskets\', query_type=\'broad\', category=\'Grocery & Gourmet Food\', 
rationale="Gourmet hampers and luxury food treats are described as a \'raving\' trend for the 2026 season (source: 
giftstoindia24x7.com)."), SearchQuery(query=\'wifi digital picture frame\', query_type=\'specific\', 
category=\'Electronics\', rationale=\'Tech gifts that foster family connection, specifically digital frames like 
Skylight, are top recommendations for 2026 (source: rushlet.blog).\'), SearchQuery(query=\'smart ring health 
tracker\', query_type=\'specific\', category=\'Wearable Technology\', rationale=\'Smart rings are projected as the 
next evolution of wearable tech for women in 2026, blending fashion with wellness monitoring (source: 
truegiftworld.com).\')]',
            name='product_research_tool',
            id='bd63fa3f-6d1c-4ab8-8ec3-c2a053c4bd25',
            tool_call_id='call_WsAHuyklC8dAse3kGqiQmDKj'
        ),
        AIMessage(
            content='',
            additi

In [47]:
resp = product_agent.invoke({"messages": [{"role": "user", "content": "what is the difference between the first and third spa set" }]}, config=config)


[values] {'messages': [HumanMessage(content="What are popular gifts for Mother's Day?", additional_kwargs={}, response_metadata={}, id='4229bb7c-6833-47ad-9f24-151d67a8ac97'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 275, 'total_tokens': 295, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-CvYfYWyq4iHGycLVYRPaTigGNhT0q', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b9b1e-194c-7023-8e89-f7768be26ff1-0', tool_calls=[{'name': 'product_research_tool', 'args': {'query': "popular gifts for Mother's Day"}, 'id': 'call_WsAHuyklC8dAse3kGqiQmDKj', 'type': 'tool_call'}], invalid_to

In [48]:
rich.print(resp['messages'][-1].content )

Here are the differences between the first and third spa gift sets:

### 1. **First Spa Bath Gift Set**
- **Title**: Spa Bath Gift Sets for Women, 8 Pcs Rose Gift Baskets
- **Price**: $17.99
- **Details**: Unfortunately, I couldn't retrieve specific details or a description for this set. However, it 
likely includes various bath and body products aimed at providing a relaxing experience. 
- **Rating**: 4.5 stars from 90 reviews.
- **Link**: [View 
More](https://www.walmart.com/ip/Spa-Bath-Gift-Sets-for-Women-8-Pcs-Rose-Scent-Gift-Baskets-for-Mom-Grandma-Sister-
Valentines-Day-Relaxing-Birthday-Gifts-for-Her/5438833867)

---

### 2. **Third Spa Bath Gift Set**
- **Title**: [Spa Bath Gift Sets for Women - 6 Pcs Rose Scent Gifts 
Box](https://www.walmart.com/ip/Spa-Bath-Gift-Sets-for-Women-6-Pcs-Rose-Scent-Gifts-Box-Relaxing-Body-Care-Holiday-
Birthday-Valentines-Day-Gifts-for-Her/340118341)
- **Brand**: Spa Luxetique
- **Price**: (Not specified in the data, but you can check the link)
- **Description**: This set includes:
  - Shower Gel: 3.21oz (95ml)
  - Bubble Bath: 3.21oz (95ml)
  - Hand Cream: 1.01oz (30ml)
  - Body Butter: 3.2oz
  - Bar Soap: 1.76oz
  - Reusable Burlap Linen Bag with Drawstring
- **Key Benefits**: Moisturizes and nourishes dry skin with ingredients like shea butter and vitamin E. The hand 
cream is rose-scented.
- **Occasion**: Ideal for Mother's Day, Birthdays, or other gifting occasions.
- **Rating**: Not specified in the retrieved details.
- **Additional Features**: Comes in a beautiful packaging, making it an appealing gift.

### Summary of Differences:
- The **first set** consists of 8 pieces and details about its contents are unclear, while the **second set** is a 
6-piece set with detailed descriptions and specific products included.
- The **second set** is branded and marketed for various situations, including Mother's Day, with clear skin care 
benefits.

If you need more specific information about the first set or any other details, please let me know!